# Root Cause Analysis

1. Aggregate transaction-level data.
To make the analysis more meaningful, I introduced controlled perturbations to simulate noticeable growth patterns.

2. Select a target merchant and benchmark key business metrics (e.g., transaction amount, transaction count) against peer merchants within the same category.

In [1]:
import pandas as pd
import numpy as np
from helper import get_generation

In [2]:
# Load dataset from huggingface
splits = {'train': 'credit_card_transaction_train.csv', 'test': 'credit_card_transaction_test.csv'}
df_all = pd.read_csv("hf://datasets/pointe77/credit-card-transaction/" + splits["train"], index_col=0)

/Users/yanxinye/github/ml-ops-notes/ml/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# ETL
df_all = df_all.assign(
    trans_date_trans_time=lambda x: pd.to_datetime(x['trans_date_trans_time']),
    trans_date=lambda x: x['trans_date_trans_time'].dt.strftime('%Y-%m-%d')
)

In [4]:
TARGET_MERCHANTS = "fraud_Kilback LLC"

# Date range of the analysis
START_DATE_TY = "2020-01-01"
START_DATE_LY = "2019-01-01"
END_DATE_TY = "2020-05-31"
END_DATE_LY = "2019-05-31"

In [5]:
df = df_all[
    (
        (df_all['trans_date'] >= START_DATE_TY) &
        (df_all['trans_date'] <= END_DATE_TY)
    )
    |
    (
        (df_all['trans_date'] >= START_DATE_LY) &
        (df_all['trans_date'] <= END_DATE_LY)
    )
].copy()

In [6]:
# Convert dob to datetime and extract generation
df = df.assign(
    dob=lambda x: pd.to_datetime(x['dob']),
    generation=lambda x: x['dob'].dt.year.map(get_generation)
)

In [7]:
df_comparison = (
    df.assign(
        yr_flag=lambda x: x['trans_date'].apply(
            lambda d: 'TY' if START_DATE_TY <= d <= END_DATE_TY
            else ('LY' if START_DATE_LY <= d <= END_DATE_LY else None)
        )
    )
    .groupby(['generation', 'state', 'gender', 'yr_flag', 'category', 'merchant'], as_index=False)['amt'].sum()
    .pivot_table(
        index=['generation', 'state', 'gender', 'category', 'merchant'],
        columns='yr_flag',
        values='amt',
        fill_value=0
    )
    .reset_index()
    .rename(columns={'TY': 'amt_ty', 'LY': 'amt_ly'})
    .rename_axis(None, axis=1)
)


In [8]:
# This data does not have any growth. Add some random variation to make it more interesting.

# Add more variation to amt_ty and amt_ly for bigger growth

np.random.seed(177)
random_factors_ty = np.clip(np.random.normal(0.3, 25, len(df_comparison)), -1, 3)
random_factors_ly = np.clip(np.random.normal(0, 2, len(df_comparison)), -0.5, 0.5)

df_comparison['amt_ty'] *= (1 + random_factors_ty)

df_comparison['amt_ly'] *= (1 + random_factors_ly)

In [15]:
def calculate_growth(df):
    df['amt_diff'] = df['amt_ty'] - df['amt_ly']
    total_amt_diff = df['amt_diff'].sum()

    # df['amt_growth'] = (
    #     (df['amt_ty'] - df['amt_ly']) 
    #     / df['amt_ly'].replace(0, pd.NA)
    # )

    total_amt_ly = df['amt_ly'].sum()
    print(f"Total amount in LY: {total_amt_ly}")
    print(f"Total amount difference: {total_amt_diff}")
    print(f"Total growth: {total_amt_diff / total_amt_ly:.2%}")
    df["amt_growth_ctc"] = 1/total_amt_ly * df["amt_diff"]

    return df

#df_comparison = calculate_growth(df_comparison)

In [16]:
# df_comparison.head()

## Target vs Peers

In [37]:
df_target = df_comparison[df_comparison['merchant'] == TARGET_MERCHANTS].copy()
df_target = calculate_growth(df_target)

Total amount in LY: 90116.45826107437
Total amount difference: 108492.6530555585
Total growth: 120.39%


In [38]:
df_peer = df_comparison[df_comparison['merchant'] != TARGET_MERCHANTS].copy()
df_peer = calculate_growth(df_peer)
df_peer.head()

Total amount in LY: 22069933.862833995
Total amount difference: 21196259.02506728
Total growth: 96.04%


,generation,state,gender,category,merchant,amt_ly,amt_ty,amt_diff,amt_growth_ctc
0,Boomer,AL,F,entertainment,fraud_Abbott-Rogahn,4.110,82.600000,78.490000,0.000004
1,Boomer,AL,F,entertainment,fraud_Abshire PLC,206.715,29.081219,-177.633781,-0.000008
2,Boomer,AL,F,entertainment,fraud_Bauch-Blanda,0.000,0.000000,0.000000,0.000000
3,Boomer,AL,F,entertainment,fraud_Beier LLC,0.000,0.000000,0.000000,0.000000
4,Boomer,AL,F,entertainment,fraud_Bins-Tillman,77.965,983.120000,905.155000,0.000041


In [44]:
peer_agg_metrics = {
    "amt_ty": ["mean"],
    "amt_ly": ["mean"],
}

peer_agg = df_peer.groupby(['generation', 'state', 'gender', 'category'])[['amt_ty', 'amt_ly']].agg(peer_agg_metrics).reset_index()

peer_agg.columns = [
    f"{col[0]}_{col[1].lower()}" if col[1] else col[0].lower()
    for col in peer_agg.columns
]


total_amt_ly_peer = peer_agg["amt_ly_mean"].sum()
peer_agg["amt_diff_mean"] = peer_agg["amt_ty_mean"] - peer_agg["amt_ly_mean"]
peer_agg["amt_growth_ctc_mean"] = peer_agg["amt_diff_mean"]/total_amt_ly_peer
print(f"Total growth for peers: {peer_agg['amt_growth_ctc_mean'].sum():.2%}")
peer_agg["merchant"] = f"ex-{TARGET_MERCHANTS}"

peer_agg.head()

Total growth for peers: 95.59%


,generation,state,gender,category,amt_ty_mean,amt_ly_mean,amt_diff_mean,amt_growth_ctc_mean,merchant
0,Boomer,AL,F,entertainment,305.651965,73.840527,231.811438,0.000442,ex-fraud_Kilback LLC
1,Boomer,AL,F,food_dining,294.358221,111.020952,183.337270,0.000350,ex-fraud_Kilback LLC
2,Boomer,AL,F,gas_transport,330.614722,158.501211,172.113510,0.000328,ex-fraud_Kilback LLC
3,Boomer,AL,F,grocery_net,24.668199,44.622558,-19.954359,-0.000038,ex-fraud_Kilback LLC
4,Boomer,AL,F,grocery_pos,385.485086,280.136657,105.348429,0.000201,ex-fraud_Kilback LLC
